# 30 requêtes SQL et leur équivalent Pandas

Notebook de l'article [30 requêtes SQL expliquées par leurs équivalents Pandas](https://moncoachdata.com/blog/requetes-sql-expliquees/) de MonCoachData. Base utilisée : [Chinook](https://github.com/lerocha/chinook-database). Code testé avec Python 3.13 et pandas 3.

In [1]:
!pip install -q duckdb

In [2]:
import os
from urllib.request import urlretrieve

if not os.path.exists("Chinook_Sqlite.sqlite"):
    urlretrieve(
        "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/requetes-sql-expliquees/Chinook_Sqlite.sqlite",
        "Chinook_Sqlite.sqlite",
    )

## Connexion à la base Chinook

Python embarque SQLite. La fonction `sql()` exécute une requête et renvoie un DataFrame.

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("Chinook_Sqlite.sqlite")

def sql(requete):
    """Exécute une requête SQL et renvoie le résultat sous forme de DataFrame."""
    return pd.read_sql_query(requete, conn)

Les 11 tables de la base.

In [4]:
tables = sql("SELECT name FROM sqlite_master WHERE type = 'table'")

for nom in tables["name"]:
    n = sql(f"SELECT COUNT(*) AS n FROM {nom}").loc[0, "n"]
    print(f"{nom:<14} {n:>5} lignes")

Album            347 lignes
Artist           275 lignes
Customer          59 lignes
Employee           8 lignes
Genre             25 lignes
Invoice          412 lignes
InvoiceLine     2240 lignes
MediaType          5 lignes
Playlist          18 lignes
PlaylistTrack   8715 lignes
Track           3503 lignes


On charge quelques tables dans des DataFrames, pour comparer SQL et Pandas.

In [5]:
customers = sql("SELECT * FROM Customer")
genres = sql("SELECT * FROM Genre")
tracks = sql("SELECT * FROM Track")
invoices = sql("SELECT * FROM Invoice")
employees = sql("SELECT * FROM Employee")

print(customers.shape, tracks.shape, invoices.shape, employees.shape, genres.shape)

(59, 13) (3503, 9) (412, 9) (8, 15) (25, 2)


## Premières lignes : SELECT, LIMIT, head()

In [6]:
sql("""
SELECT * FROM Genre
LIMIT 5
""")

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll


In [7]:
genres.head()

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll


Sélectionner des colonnes.

In [8]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
LIMIT 10
""")

,Name,Composer,UnitPrice
0,For Those About To Rock (We Salute You),"Angus Young, Malcolm Young, Brian Johnson",0.99
1,Balls to the Wall,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",0.99
2,Fast As a Shark,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",0.99
3,Restless and Wild,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",0.99
4,Princess of the Dawn,Deaffy & R.A. Smith-Diesel,0.99
5,Put The Finger On You,"Angus Young, Malcolm Young, Brian Johnson",0.99
6,Let's Get It Up,"Angus Young, Malcolm Young, Brian Johnson",0.99
7,Inject The Venom,"Angus Young, Malcolm Young, Brian Johnson",0.99
8,Snowballed,"Angus Young, Malcolm Young, Brian Johnson",0.99
9,Evil Walks,"Angus Young, Malcolm Young, Brian Johnson",0.99


In [9]:
tracks[["Name", "Composer", "UnitPrice"]].head(10)

,Name,Composer,UnitPrice
0,For Those About To Rock (We Salute You),"Angus Young, Malcolm Young, Brian Johnson",0.99
1,Balls to the Wall,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",0.99
2,Fast As a Shark,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",0.99
3,Restless and Wild,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",0.99
4,Princess of the Dawn,Deaffy & R.A. Smith-Diesel,0.99
5,Put The Finger On You,"Angus Young, Malcolm Young, Brian Johnson",0.99
6,Let's Get It Up,"Angus Young, Malcolm Young, Brian Johnson",0.99
7,Inject The Venom,"Angus Young, Malcolm Young, Brian Johnson",0.99
8,Snowballed,"Angus Young, Malcolm Young, Brian Johnson",0.99
9,Evil Walks,"Angus Young, Malcolm Young, Brian Johnson",0.99


## DISTINCT et unique()

In [10]:
sql("""
SELECT DISTINCT FirstName
FROM Employee
""")

,FirstName
0,Andrew
1,Nancy
2,Jane
3,Margaret
4,Steve
5,Michael
6,Robert
7,Laura


In [11]:
employees["FirstName"].unique().tolist()

['Andrew', 'Nancy', 'Jane', 'Margaret', 'Steve', 'Michael', 'Robert', 'Laura']

## Compter : COUNT

In [12]:
sql("""
SELECT COUNT(*) FROM Track
""")

,COUNT(*)
0,3503


In [13]:
tracks.shape[0]

3503

Compter les valeurs distinctes.

In [14]:
sql("""
SELECT COUNT(DISTINCT FirstName) FROM Employee
""")

,COUNT(DISTINCT FirstName)
0,8


In [15]:
employees["FirstName"].nunique()

8

## Filtrer : WHERE et loc[]

In [16]:
sql("""
SELECT TrackId, Name, UnitPrice
FROM Track
WHERE UnitPrice > 1.0
LIMIT 10
""")

,TrackId,Name,UnitPrice
0,2819,Battlestar Galactica: The Story So Far,1.99
1,2820,Occupation / Precipice,1.99
2,2821,"Exodus, Pt. 1",1.99
3,2822,"Exodus, Pt. 2",1.99
4,2823,Collaborators,1.99
5,2824,Torn,1.99
6,2825,A Measure of Salvation,1.99
7,2826,Hero,1.99
8,2827,Unfinished Business,1.99
9,2828,The Passage,1.99


In [17]:
tracks.loc[tracks["UnitPrice"] > 1.0, ["TrackId", "Name", "UnitPrice"]].head(10)

,TrackId,Name,UnitPrice
2818,2819,Battlestar Galactica: The Story So Far,1.99
2819,2820,Occupation / Precipice,1.99
2820,2821,"Exodus, Pt. 1",1.99
2821,2822,"Exodus, Pt. 2",1.99
2822,2823,Collaborators,1.99
2823,2824,Torn,1.99
2824,2825,A Measure of Salvation,1.99
2825,2826,Hero,1.99
2826,2827,Unfinished Business,1.99
2827,2828,The Passage,1.99


Plusieurs conditions : AND devient `&` (chaque condition entre parenthèses).

In [18]:
sql("""
SELECT COUNT(*) FROM Track
WHERE UnitPrice > 1.0 AND UnitPrice < 10
""")

,COUNT(*)
0,213


In [19]:
len(tracks[(tracks["UnitPrice"] > 1) & (tracks["UnitPrice"] < 10)])

213

OR devient `|`.

In [20]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity = 'Paris' OR BillingCity = 'Berlin'
LIMIT 5
""")

,BillingAddress,BillingCity,Total
0,Barbarossastraße 19,Berlin,1.98
1,"8, Rue Hanovre",Paris,1.98
2,"8, Rue Hanovre",Paris,13.86
3,Tauentzienstraße 8,Berlin,1.98
4,Barbarossastraße 19,Berlin,3.96


In [21]:
villes = (invoices["BillingCity"] == "Paris") | (invoices["BillingCity"] == "Berlin")

invoices.loc[villes, ["BillingAddress", "BillingCity", "Total"]].head(5)

,BillingAddress,BillingCity,Total
6,Barbarossastraße 19,Berlin,1.98
7,"8, Rue Hanovre",Paris,1.98
18,"8, Rue Hanovre",Paris,13.86
28,Tauentzienstraße 8,Berlin,1.98
29,Barbarossastraße 19,Berlin,3.96


`!=` et `<>` sont équivalents en SQL.

In [22]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity != 'Paris' AND BillingCity <> 'Berlin'
LIMIT 5
""")

,BillingAddress,BillingCity,Total
0,Theodor-Heuss-Straße 34,Stuttgart,1.98
1,Ullevålsveien 14,Oslo,3.96
2,Grétrystraat 63,Brussels,5.94
3,8210 111 ST NW,Edmonton,8.91
4,69 Salem Street,Boston,13.86


In [23]:
autres = (invoices["BillingCity"] != "Paris") & (invoices["BillingCity"] != "Berlin")

invoices.loc[autres, ["BillingAddress", "BillingCity", "Total"]].head(5)

,BillingAddress,BillingCity,Total
0,Theodor-Heuss-Straße 34,Stuttgart,1.98
1,Ullevålsveien 14,Oslo,3.96
2,Grétrystraat 63,Brussels,5.94
3,8210 111 ST NW,Edmonton,8.91
4,69 Salem Street,Boston,13.86


## IN et BETWEEN

In [24]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity IN ('Berlin', 'Paris', 'New York', 'Boston', 'London')
LIMIT 5
""")

,BillingAddress,BillingCity,Total
0,69 Salem Street,Boston,13.86
1,Barbarossastraße 19,Berlin,1.98
2,"8, Rue Hanovre",Paris,1.98
3,202 Hoxton Street,London,8.91
4,"8, Rue Hanovre",Paris,13.86


In [25]:
cinq_villes = ["Berlin", "Paris", "New York", "Boston", "London"]

invoices.loc[invoices["BillingCity"].isin(cinq_villes), ["BillingAddress", "BillingCity", "Total"]].head(5)

,BillingAddress,BillingCity,Total
4,69 Salem Street,Boston,13.86
6,Barbarossastraße 19,Berlin,1.98
7,"8, Rue Hanovre",Paris,1.98
10,202 Hoxton Street,London,8.91
18,"8, Rue Hanovre",Paris,13.86


NOT IN devient `~isin()`.

In [26]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity NOT IN ('Berlin', 'Paris', 'New York', 'Boston', 'London')
LIMIT 5
""")

,BillingAddress,BillingCity,Total
0,Theodor-Heuss-Straße 34,Stuttgart,1.98
1,Ullevålsveien 14,Oslo,3.96
2,Grétrystraat 63,Brussels,5.94
3,8210 111 ST NW,Edmonton,8.91
4,Berger Straße 10,Frankfurt,0.99


In [27]:
invoices.loc[~invoices["BillingCity"].isin(cinq_villes), ["BillingAddress", "BillingCity", "Total"]].head(5)

,BillingAddress,BillingCity,Total
0,Theodor-Heuss-Straße 34,Stuttgart,1.98
1,Ullevålsveien 14,Oslo,3.96
2,Grétrystraat 63,Brussels,5.94
3,8210 111 ST NW,Edmonton,8.91
5,Berger Straße 10,Frankfurt,0.99


BETWEEN inclut les deux bornes.

In [28]:
sql("""
SELECT BillingCity, Total
FROM Invoice
WHERE Total BETWEEN 5 AND 15
LIMIT 5
""")

,BillingCity,Total
0,Brussels,5.94
1,Edmonton,8.91
2,Boston,13.86
3,Dublin,5.94
4,London,8.91


In [29]:
invoices.loc[invoices["Total"].between(5, 15), ["BillingCity", "Total"]].head(5)

,BillingCity,Total
2,Brussels,5.94
3,Edmonton,8.91
4,Boston,13.86
9,Dublin,5.94
10,London,8.91


## Valeurs manquantes : IS NULL et isnull()

In [30]:
sql("""
SELECT COUNT(*) FROM Invoice
WHERE BillingState IS NULL
""")

,COUNT(*)
0,202


In [31]:
int(invoices["BillingState"].isnull().sum())

202

IS NOT NULL devient `notna()`.

In [32]:
sql("""
SELECT InvoiceDate, BillingCountry
FROM Invoice
WHERE BillingState IS NOT NULL
LIMIT 10
""")

,InvoiceDate,BillingCountry
0,2021-01-06 00:00:00,Canada
1,2021-01-11 00:00:00,USA
2,2021-02-03 00:00:00,Ireland
3,2021-02-19 00:00:00,USA
4,2021-03-04 00:00:00,USA
5,2021-03-04 00:00:00,USA
6,2021-03-05 00:00:00,USA
7,2021-03-06 00:00:00,USA
8,2021-03-09 00:00:00,Canada
9,2021-04-04 00:00:00,Australia


In [33]:
invoices.loc[invoices["BillingState"].notna(), ["InvoiceDate", "BillingCountry"]].head(10)

,InvoiceDate,BillingCountry
3,2021-01-06 00:00:00,Canada
4,2021-01-11 00:00:00,USA
9,2021-02-03 00:00:00,Ireland
12,2021-02-19 00:00:00,USA
13,2021-03-04 00:00:00,USA
14,2021-03-04 00:00:00,USA
15,2021-03-05 00:00:00,USA
16,2021-03-06 00:00:00,USA
17,2021-03-09 00:00:00,Canada
20,2021-04-04 00:00:00,Australia


## LIKE et les méthodes `str`

In [34]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE 'B%'
LIMIT 5
""")

,Name,Composer,UnitPrice
0,Balls to the Wall,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",0.99
1,Breaking The Rules,"Angus Young, Malcolm Young, Brian Johnson",0.99
2,Bad Boy Boogie,AC/DC,0.99
3,Blind Man,"Steven Tyler, Joe Perry, Taylor Rhodes",0.99
4,Bleed The Freak,Jerry Cantrell,0.99


In [35]:
tracks.loc[tracks["Name"].str.startswith("B"), ["Name", "Composer", "UnitPrice"]].head(5)

,Name,Composer,UnitPrice
1,Balls to the Wall,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",0.99
11,Breaking The Rules,"Angus Young, Malcolm Young, Brian Johnson",0.99
17,Bad Boy Boogie,AC/DC,0.99
30,Blind Man,"Steven Tyler, Joe Perry, Taylor Rhodes",0.99
53,Bleed The Freak,Jerry Cantrell,0.99


Piège : dans SQLite, LIKE ignore la casse. `str.contains()` la respecte.

In [36]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE '%beautiful%'
""")

,Name,Composer,UnitPrice
0,So Beautiful,Mick Hucknall,0.99
1,Beautiful Day,"Adam Clayton, Bono, Larry Mullen, The Edge",0.99
2,Beautiful Boy,None,0.99
3,On the Beautiful Blue Danube,Johann Strauss II,0.99


In [37]:
contient = tracks["Name"].str.contains("beautiful")

print("sans case=False :", contient.sum(), "titre(s) trouvé(s)")
print("avec case=False :", tracks["Name"].str.contains("beautiful", case=False).sum(), "titre(s) trouvé(s)")

sans case=False : 0 titre(s) trouvé(s)
avec case=False : 4 titre(s) trouvé(s)


Avec `case=False`, on obtient le même résultat.

In [38]:
tracks.loc[tracks["Name"].str.contains("beautiful", case=False), ["Name", "Composer", "UnitPrice"]]

,Name,Composer,UnitPrice
2445,So Beautiful,Mick Hucknall,0.99
2937,Beautiful Day,"Adam Clayton, Bono, Larry Mullen, The Edge",0.99
3267,Beautiful Boy,None,0.99
3444,On the Beautiful Blue Danube,Johann Strauss II,0.99


LIKE combiné à une autre condition.

In [39]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE 'F%' AND UnitPrice > 1.0
LIMIT 5
""")

,Name,Composer,UnitPrice
0,Fallout,None,1.99
1,Further Instructions,None,1.99
2,Flashes Before Your Eyes,None,1.99
3,Fire + Water,None,1.99
4,Five Years Gone,None,1.99


In [40]:
tracks.loc[
    tracks["Name"].str.startswith("F") & (tracks["UnitPrice"] > 1.0),
    ["Name", "Composer", "UnitPrice"],
].head(5)

,Name,Composer,UnitPrice
2848,Fallout,None,1.99
2862,Further Instructions,None,1.99
2880,Flashes Before Your Eyes,None,1.99
2891,Fire + Water,None,1.99
3166,Five Years Gone,None,1.99


## Agrégations : SUM, MAX, MIN, AVG

In [41]:
sql("""
SELECT SUM(Total), MAX(Total), MIN(Total), AVG(Total)
FROM Invoice
""")

,SUM(Total),MAX(Total),MIN(Total),AVG(Total)
0,2328.6,25.86,0.99,5.651942


In [42]:
invoices["Total"].agg(["sum", "max", "min", "mean"])

,Total
sum,2328.600000
max,25.860000
min,0.990000
mean,5.651942


Piège : une colonne non agrégée sans GROUP BY. SQLite accepte, avec une valeur arbitraire.

In [43]:
sql("""
SELECT AVG(Total), BillingCity, BillingAddress
FROM Invoice
""")

,AVG(Total),BillingCity,BillingAddress
0,5.651942,Stuttgart,Theodor-Heuss-Straße 34


DuckDB refuse la requête.

In [44]:
import duckdb

try:
    duckdb.sql("SELECT AVG(Total), BillingCity FROM invoices").df()
except duckdb.Error as erreur:
    print(str(erreur).splitlines()[0])

Binder Error: column "BillingCity" must appear in the GROUP BY clause or must be part of an aggregate function.


Agrégation sur un sous-ensemble.

In [45]:
sql("""
SELECT AVG(Total) AS moyenne
FROM Invoice
WHERE BillingCity = 'Paris'
""")

,moyenne
0,5.517143


In [46]:
float(invoices.loc[invoices["BillingCity"] == "Paris", "Total"].mean())

5.517142857142857

## Calculs entre colonnes, division entière et alias

In [47]:
sql("""
SELECT SUM(Total) / COUNT(Total)
FROM Invoice
""")

,SUM(Total) / COUNT(Total)
0,5.651942


In [48]:
float(invoices["Total"].sum() / invoices["Total"].count())

5.651941747572814

Piège : en SQL, la division de deux entiers donne un entier.

In [49]:
sql("""
SELECT 10 / 3
""")

,10 / 3
0,3


In [50]:
print(10 // 3)    # division entière, comme SQL
print(10 / 3)     # division décimale, comme Python

3
3.3333333333333335


Avec un nombre décimal, on obtient un résultat décimal.

In [51]:
sql("""
SELECT 10.0 / 3.0
""")

,10.0 / 3.0
0,3.333333


Durée des titres en minutes.

In [52]:
sql("""
SELECT Milliseconds / 1000.0 / 60.0
FROM Track
LIMIT 10
""")

,Milliseconds / 1000.0 / 60.0
0,5.728650
1,5.709367
2,3.843650
3,4.200850
4,6.256967
5,3.427700
6,3.898767
7,3.513900
8,3.385033
9,4.391617


In [53]:
tracks["Milliseconds"].head(10) / 1000 / 60

,Milliseconds
0,5.728650
1,5.709367
2,3.843650
3,4.200850
4,6.256967
5,3.427700
6,3.898767
7,3.513900
8,3.385033
9,4.391617


Alias avec AS. SQLite affiche 2328.6 et Pandas 2328.5999999999995 (arithmétique flottante).

In [54]:
sql("""
SELECT SUM(Total) AS sum_total
FROM Invoice
""")

,sum_total
0,2328.6


In [55]:
float(invoices["Total"].sum())

2328.5999999999995

Plusieurs agrégats nommés.

In [56]:
sql("""
SELECT SUM(Total) AS sum_total,
       MAX(Total) AS max_total,
       MIN(Total) AS min_total,
       SUM(Total) / COUNT(Total) AS mean_total
FROM Invoice
""")

,sum_total,max_total,min_total,mean_total
0,2328.6,25.86,0.99,5.651942


In [57]:
invoices["Total"].agg(sum_total="sum", max_total="max", min_total="min", mean_total="mean")

,Total
sum_total,2328.600000
max_total,25.860000
min_total,0.990000
mean_total,5.651942


## Trier : ORDER BY et sort_values()

In [58]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY Composer, TrackId
LIMIT 10
""")

,Name,Composer,UnitPrice
0,Iron Man,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
1,Children Of The Grave,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2,Paranoid,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
3,New Rhumba,A. Jamal,0.99
4,Astronomy,A.Bouchard/J.Bouchard/S.Pearlman,0.99
5,Hard To Handle,A.Isbell/A.Jones/O.Redding,0.99
6,Go Down,AC/DC,0.99
7,Dog Eat Dog,AC/DC,0.99
8,Let There Be Rock,AC/DC,0.99
9,Bad Boy Boogie,AC/DC,0.99


In [59]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["Composer", "TrackId"])[["Name", "Composer", "UnitPrice"]]
    .head(10))

,Name,Composer,UnitPrice
2106,Iron Man,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2107,Children Of The Grave,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2108,Paranoid,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
1907,New Rhumba,A. Jamal,0.99
414,Astronomy,A.Bouchard/J.Bouchard/S.Pearlman,0.99
2588,Hard To Handle,A.Isbell/A.Jones/O.Redding,0.99
14,Go Down,AC/DC,0.99
15,Dog Eat Dog,AC/DC,0.99
16,Let There Be Rock,AC/DC,0.99
17,Bad Boy Boogie,AC/DC,0.99


Piège : SQLite place les NULL en premier, Pandas place les NaN en dernier.

In [60]:
sql("""
SELECT TrackId, Composer
FROM Track
ORDER BY Composer, TrackId
LIMIT 3
""")

,TrackId,Composer
0,63,None
1,64,None
2,65,None


In [61]:
print(tracks.sort_values(["Composer", "TrackId"])[["TrackId", "Composer"]].head(3))
print()
print(tracks.sort_values(["Composer", "TrackId"], na_position="first")[["TrackId", "Composer"]].head(3))

      TrackId                                      Composer
2106     2107  A. F. Iommi, W. Ward, T. Butler, J. Osbourne
2107     2108  A. F. Iommi, W. Ward, T. Butler, J. Osbourne
2108     2109  A. F. Iommi, W. Ward, T. Butler, J. Osbourne

    TrackId Composer
62       63     None
63       64     None
64       65     None


Tri sur plusieurs colonnes.

In [62]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY UnitPrice, Composer, Name
LIMIT 10
""")

,Name,Composer,UnitPrice
0,Children Of The Grave,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
1,Iron Man,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2,Paranoid,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
3,New Rhumba,A. Jamal,0.99
4,Astronomy,A.Bouchard/J.Bouchard/S.Pearlman,0.99
5,Hard To Handle,A.Isbell/A.Jones/O.Redding,0.99
6,Bad Boy Boogie,AC/DC,0.99
7,Dog Eat Dog,AC/DC,0.99
8,Go Down,AC/DC,0.99
9,Hell Ain't A Bad Place To Be,AC/DC,0.99


In [63]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["UnitPrice", "Composer", "Name"])[["Name", "Composer", "UnitPrice"]]
    .head(10))

,Name,Composer,UnitPrice
2107,Children Of The Grave,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2106,Iron Man,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
2108,Paranoid,"A. F. Iommi, W. Ward, T. Butler, J. Osbourne",0.99
1907,New Rhumba,A. Jamal,0.99
414,Astronomy,A.Bouchard/J.Bouchard/S.Pearlman,0.99
2588,Hard To Handle,A.Isbell/A.Jones/O.Redding,0.99
17,Bad Boy Boogie,AC/DC,0.99
15,Dog Eat Dog,AC/DC,0.99
14,Go Down,AC/DC,0.99
20,Hell Ain't A Bad Place To Be,AC/DC,0.99


Tri décroissant (`DESC`) sur certaines colonnes.

In [64]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY UnitPrice DESC, Composer DESC, Name ASC
LIMIT 10
""")

,Name,Composer,UnitPrice
0,A Twist In The Tail,roger glover,0.99
1,Lick It Up,roger glover,0.99
2,One Man's Meat,roger glover,0.99
3,Ramshackle Man,roger glover,0.99
4,Solitaire,roger glover,0.99
5,Talk About Love,roger glover,0.99
6,Time To Kill,roger glover,0.99
7,Loves Been Good To Me,rod mckuen,0.99
8,For Once In My Life,orlando murden/ronald miller,0.99
9,The Lady Is A Tramp,lorenz hart/richard rodgers,0.99


In [65]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["UnitPrice", "Composer", "Name"], ascending=[False, False, True])[["Name", "Composer", "UnitPrice"]]
    .head(10))

,Name,Composer,UnitPrice
821,A Twist In The Tail,roger glover,0.99
816,Lick It Up,roger glover,0.99
824,One Man's Meat,roger glover,0.99
820,Ramshackle Man,roger glover,0.99
823,Solitaire,roger glover,0.99
818,Talk About Love,roger glover,0.99
819,Time To Kill,roger glover,0.99
1054,Loves Been Good To Me,rod mckuen,0.99
1040,For Once In My Life,orlando murden/ronald miller,0.99
1051,The Lady Is A Tramp,lorenz hart/richard rodgers,0.99


## Regrouper : GROUP BY et groupby()

In [66]:
sql("""
SELECT GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
LIMIT 10
""")

,GenreId,nb_titres
0,1,1297
1,2,130
2,3,374
3,4,332
4,5,12
5,6,81
6,7,579
7,8,58
8,9,48
9,10,43


In [67]:
tracks.groupby("GenreId").size().rename("nb_titres").reset_index().head(10)

,GenreId,nb_titres
0,1,1297
1,2,130
2,3,374
3,4,332
4,5,12
5,6,81
6,7,579
7,8,58
8,9,48
9,10,43


Piège : une colonne non groupée. SQLite renvoie un nom quelconque du groupe.

In [68]:
sql("""
SELECT Name, GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
LIMIT 5
""")

,Name,GenreId,nb_titres
0,For Those About To Rock (We Salute You),1,1297
1,Desafinado,2,130
2,Enter Sandman,3,374
3,Your Time Has Come,4,332
4,Money,5,12


DuckDB refuse la requête.

In [69]:
try:
    duckdb.sql("SELECT Name, GenreId, COUNT(*) AS nb_titres FROM tracks GROUP BY GenreId").df()
except duckdb.Error as erreur:
    print(str(erreur).splitlines()[0])

Binder Error: column "Name" must appear in the GROUP BY clause or must be part of an aggregate function.


Plusieurs colonnes de groupe et plusieurs agrégats.

In [70]:
sql("""
SELECT GenreId, AlbumId,
       COUNT(*) AS nb_titres,
       AVG(Milliseconds) / 1000.0 / 60.0 AS duree_moyenne,
       AVG(UnitPrice) AS prix_moyen
FROM Track
GROUP BY GenreId, AlbumId
LIMIT 10
""")

,GenreId,AlbumId,nb_titres,duree_moyenne,prix_moyen
0,1,1,10,4.000692,0.99
1,1,2,1,5.709367,0.99
2,1,3,3,4.767156,0.99
3,1,4,8,5.110956,0.99
4,1,5,15,4.901899,0.99
5,1,6,13,4.424263,0.99
6,1,7,12,4.513007,0.99
7,1,10,14,4.675849,0.99
8,1,30,14,5.345143,0.99
9,1,31,9,4.566535,0.99


In [71]:
(tracks.groupby(["GenreId", "AlbumId"])
    .agg(
        nb_titres=("TrackId", "count"),
        duree_moyenne=("Milliseconds", lambda ms: ms.mean() / 1000 / 60),
        prix_moyen=("UnitPrice", "mean"),
    )
    .reset_index()
    .head(10))

,GenreId,AlbumId,nb_titres,duree_moyenne,prix_moyen
0,1,1,10,4.000692,0.99
1,1,2,1,5.709367,0.99
2,1,3,3,4.767156,0.99
3,1,4,8,5.110956,0.99
4,1,5,15,4.901899,0.99
5,1,6,13,4.424263,0.99
6,1,7,12,4.513007,0.99
7,1,10,14,4.675849,0.99
8,1,30,14,5.345143,0.99
9,1,31,9,4.566535,0.99


Tri des groupes sur le résultat d'un agrégat.

In [72]:
sql("""
SELECT GenreId, AlbumId,
       COUNT(*) AS nb_titres,
       AVG(Milliseconds) / 1000.0 / 60.0 AS duree_moyenne,
       AVG(UnitPrice) AS prix_moyen
FROM Track
GROUP BY GenreId, AlbumId
ORDER BY duree_moyenne DESC, nb_titres DESC
LIMIT 10
""")

,GenreId,AlbumId,nb_titres,duree_moyenne,prix_moyen
0,19,227,5,52.615260,1.99
1,20,253,24,48.759572,1.99
2,20,227,2,45.771458,1.99
3,21,229,22,45.605561,1.99
4,19,231,16,44.051442,1.99
5,18,227,12,43.763733,1.99
6,21,231,8,43.750515,1.99
7,18,226,1,43.704167,1.99
8,19,229,4,43.609338,1.99
9,19,228,3,43.447139,1.99


In [73]:
(tracks.groupby(["GenreId", "AlbumId"])
    .agg(
        nb_titres=("TrackId", "count"),
        duree_moyenne=("Milliseconds", lambda ms: ms.mean() / 1000 / 60),
        prix_moyen=("UnitPrice", "mean"),
    )
    .reset_index()
    .sort_values(["duree_moyenne", "nb_titres"], ascending=False)
    .head(10))

,GenreId,AlbumId,nb_titres,duree_moyenne,prix_moyen
265,19,227,5,52.615260,1.99
276,20,253,24,48.759572,1.99
275,20,227,2,45.771458,1.99
278,21,229,22,45.605561,1.99
269,19,231,16,44.051442,1.99
264,18,227,12,43.763733,1.99
279,21,231,8,43.750515,1.99
263,18,226,1,43.704167,1.99
267,19,229,4,43.609338,1.99
266,19,228,3,43.447139,1.99


## Filtrer les groupes : HAVING

WHERE ne peut pas filtrer un agrégat.

In [74]:
import sqlite3

requetes = [
    "SELECT GenreId FROM Track GROUP BY GenreId WHERE COUNT(GenreId) > 100",
    "SELECT GenreId FROM Track WHERE COUNT(GenreId) > 100 GROUP BY GenreId",
]

for requete in requetes:
    try:
        conn.execute(requete)
    except sqlite3.Error as erreur:
        print(f"{type(erreur).__name__} : {erreur}")

OperationalError : near "WHERE": syntax error
OperationalError : misuse of aggregate: COUNT()


La clause qui filtre les groupes est HAVING.

In [75]:
sql("""
SELECT GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
HAVING COUNT(*) > 100
""")

,GenreId,nb_titres
0,1,1297
1,2,130
2,3,374
3,4,332
4,7,579


In [76]:
nb_titres = tracks.groupby("GenreId").size().rename("nb_titres")

nb_titres[nb_titres > 100].reset_index()

,GenreId,nb_titres
0,1,1297
1,2,130
2,3,374
3,4,332
4,7,579


## Joindre deux tables : JOIN et merge()

In [77]:
sql("""
SELECT Track.Name AS titre, Genre.Name AS genre
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
LIMIT 5
""")

,titre,genre
0,For Those About To Rock (We Salute You),Rock
1,Balls to the Wall,Rock
2,Fast As a Shark,Rock
3,Restless and Wild,Rock
4,Princess of the Dawn,Rock


In [78]:
(tracks.merge(genres, on="GenreId", suffixes=("_titre", "_genre"))
    [["Name_titre", "Name_genre"]]
    .rename(columns={"Name_titre": "titre", "Name_genre": "genre"})
    .head(5))

,titre,genre
0,For Those About To Rock (We Salute You),Rock
1,Balls to the Wall,Rock
2,Fast As a Shark,Rock
3,Restless and Wild,Rock
4,Princess of the Dawn,Rock


## SQL directement sur un DataFrame avec DuckDB

In [79]:
duckdb.sql("""
    SELECT GenreId, COUNT(*) AS nb_titres
    FROM tracks
    GROUP BY GenreId
    HAVING COUNT(*) > 100
    ORDER BY GenreId
""").df()

,GenreId,nb_titres
0,1,1297
1,2,130
2,3,374
3,4,332
4,7,579
